# 集群平台、网络存储与可靠性补充线 · 第 6/8 课：容量、配额、公平与 Backfill

> 状态：**参考答案版**  
> 逐课通过制：未完成代码、问答与边界解释前，不进入下一课。

## 本课目标与完成标准

本课产出：实现不延误保留作业的 backfill 选择，并解释利用率、公平性与碎片的冲突。

通过要求：唯一代码填空题通过给定检查；Q1～Q3 都给出因果链；能指出一个正确性边界和一个性能/运营取舍；总分至少 8/10。

## 与既有六条主线的边界

上一课解决单个 gang 放置；本课处理多租户队列、配额、优先级、reservation 与空洞利用。

前置：Linux/网络基础、runtime 补充线、train 分布式章节。本课只补 AI Infra 缺口，不重复已经完成的 kernel 或并行算法推导。

## 核心心智模型

### 它是什么、解决什么问题

调度器先为队首大作业估算最早启动保留点；短作业只有在资源和预计时长都不跨过保留点时才 backfill。配额/公平份额限制租户长期占用。

### 数据与控制如何流动

每轮先按公平份额/优先级确定队首并建立未来 reservation，再在当前空洞中筛选资源足够且能在保留点前结束的候选；提交后持续校准 walltime，必要抢占时把 checkpoint 成本计入收益。

### 正确性条件与常见误区

用户估计时长不准会延误 reservation；抢占必须考虑 checkpoint/终止成本和级联重试。利用率高不等于有价值工作多。

### 性能、成本与工程取舍

小作业填洞提高利用率，但频繁抢占/碎片会损害大 gang；保守 reservation 公平可预测，却可能让资源短时空闲。

## 具体演示

空闲 8 GPU、距大作业保留 30 分钟：候选 A=4GPU/20min 可 backfill，B=8GPU/40min 不可，C=10GPU/5min 资源不足。

请先独立复述“输入 → 状态变化 → 输出/指标”，再开始填空。

## 实践任务：唯一代码填空题

补齐确定性 backfill：选择可完成候选中 GPU-time 最大者。

只能修改 `TODO`/`______` 位置，不得删除断言或放宽通过条件。

In [ ]:
def choose_backfill(candidates, free_gpus, gap_minutes):
    """candidate=(name, gpus, minutes)"""
    feasible = [job for job in candidates
                if job[1] <= free_gpus and job[2] <= gap_minutes]
    if not feasible:
        return None
    # TODO：最大化 gpus*minutes；平局按名称最小。
    return ______

jobs = [("a", 4, 20), ("b", 8, 40), ("c", 2, 25)]
assert choose_backfill(jobs, 8, 30) == ("a", 4, 20)
assert choose_backfill(jobs, 1, 30) is None


### 检查方法

运行本单元格，所有断言必须通过；再补一个边界输入并解释预期。

### Q1

为什么按 GPU 数最少优先不一定提高整体利用率？

**你的答案：**


### Q2

用户低估作业时长，backfill 会造成什么？如何防护？

**你的答案：**


### Q3

高优先级抢占低优先级作业前应计算哪些成本？

**你的答案：**


## 评分规则

- 代码 4 分：正常输入 2 分、边界输入 1 分、解释 1 分；
- Q1～Q3 各 2 分；
- 一票否决：混淆测量与推测、忽略租户/请求隔离、用平均值掩盖尾延迟、删除失败路径。

## 参考答案（仅 answer 分支）

先独立完成。核对后请改变一个规模或故障条件重新推演。

In [ ]:
def choose_backfill(candidates, free_gpus, gap_minutes):
    """candidate=(name, gpus, minutes)"""
    feasible = [job for job in candidates
                if job[1] <= free_gpus and job[2] <= gap_minutes]
    if not feasible:
        return None
    return min(feasible, key=lambda job: (-job[1] * job[2], job[0]))

jobs = [("a", 4, 20), ("b", 8, 40), ("c", 2, 25)]
assert choose_backfill(jobs, 8, 30) == ("a", 4, 20)
assert choose_backfill(jobs, 1, 30) is None


### Q1 参考答案

可能选择许多很短/很小但 GPU-time 低的作业，留下无法拼成大 gang 的碎片；还忽略运行时长、拓扑和队列公平。应围绕目标函数与 reservation 约束。

### Q2 参考答案

作业跨过大 gang 的保留点，延误队首和破坏公平。可要求可信 walltime、设置安全余量、可抢占 checkpoint、惩罚反复超时，并基于历史校准。

### Q3 参考答案

checkpoint 时间/带宽、已完成有用工作、重启和数据恢复、依赖服务、级联网络/存储峰值及 SLO 收益。抢占不是零成本释放 GPU。

## 参考资料

- [Kubernetes Scheduling Framework](https://kubernetes.io/docs/concepts/scheduling-eviction/scheduling-framework/)
- [Kubernetes Gang Scheduling](https://kubernetes.io/docs/concepts/scheduling-eviction/gang-scheduling/)

API 与平台能力会演进；部署前应按目标版本重新核对。